### Code to avoid sudden kill when working with sklearn and torch simultaniuosly

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

### Libraries

In [ ]:
# stantard libs
import numpy as np
import pandas as pd

# mac optimized ML and vectorized computing
import mlx.core as mx
import mlx.nn as mnn
import mlx.optimizers as optim

# plotting
import plotly.graph_objects as go

# dataset
from sklearn.datasets import load_iris

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
from sklearn.preprocessing import StandardScaler

# torch
import torch
import torch.nn as nn
import torch.optim as optim

In [7]:


# 1. Load Data and Filter for binary classification (Setosa = 0, Versicolor = 1)
iris = load_iris(as_frame=True)
df = iris.frame
binary_df = df[df['target'] < 2]

# Select 3 features for 3D plotting
X_3d = binary_df.iloc[:, [0, 1, 2]].values  # Sepal Length, Sepal Width, Petal Length
y_binary = binary_df['target'].values

# 2. Train Perceptron
clf = Perceptron(max_iter=1000, random_state=42)
clf.fit(X_3d, y_binary)

# Extract weights and bias to calculate the hyperplane
w = clf.coef_[0]
b = clf.intercept_[0]

# 3. Create a meshgrid for the plane
x_min, x_max = X_3d[:, 0].min(), X_3d[:, 0].max()
y_min, y_max = X_3d[:, 1].min(), X_3d[:, 1].max()

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 10),
                     np.linspace(y_min, y_max, 10))

# Calculate corresponding z values for the hyperplane
# w[0]*x + w[1]*y + w[2]*z + b = 0  =>  z = -(w[0]*x + w[1]*y + b) / w[2]
zz = -(w[0] * xx + w[1] * yy + b) / w[2]

# 4. Build the Interactive Plotly Figure
fig = go.Figure()

# Add Setosa scatter points
fig.add_trace(go.Scatter3d(
    x=X_3d[y_binary == 0, 0],
    y=X_3d[y_binary == 0, 1],
    z=X_3d[y_binary == 0, 2],
    mode='markers',
    marker=dict(size=6, color='pink', opacity=0.8),
    name='Setosa'
))

# Add Versicolor scatter points
fig.add_trace(go.Scatter3d(
    x=X_3d[y_binary == 1, 0],
    y=X_3d[y_binary == 1, 1],
    z=X_3d[y_binary == 1, 2],
    mode='markers',
    marker=dict(size=6, color='green', opacity=0.8),
    name='Versicolor'
))

# Add the separating hyperplane
fig.add_trace(go.Surface(
    x=xx, y=yy, z=zz,
    colorscale='Greys',
    opacity=0.7,
    showscale=False,
    name='Hyperplane'
))

# Update layout for clean axes and titles
fig.update_layout(
    title='Interactive 3D Hyperplane (Perceptron)',
    scene=dict(
        xaxis_title='Sepal Length',
        yaxis_title='Sepal Width',
        zaxis_title='Petal Length'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# Render the plot
fig.show()


In [8]:


# 1. Prepare data
iris = load_iris(as_frame=True)
X_full, y_full = iris.frame.iloc[:, :4].values, iris.frame['target'].values
target_names = iris.target_names

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_full, test_size=0.2, random_state=42)

# Convert to MLX arrays (both inputs and targets)
X_train_mx = mx.array(X_train, dtype=mx.float32)
y_train_mx = mx.array(y_train, dtype=mx.int32)
X_test_mx = mx.array(X_test, dtype=mx.float32)

# 2. Define MLX Architecture
class IrisMLP(mnn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = mnn.Linear(4, 8)
        self.output = mnn.Linear(8, 3)

    def __call__(self, x):
        # We output RAW LOGITS here for training stability.
        # We will apply softmax later for the final predictions.
        x = mnn.relu(self.hidden(x))
        logits = self.output(x)
        return logits

model = IrisMLP()
mx.eval(model.parameters())

# 3. Setup Optimizer and Loss Function
optimizer = optim.Adam(learning_rate=0.01)

def loss_fn(model, X, y):
    logits = model(X)
    # MLX's cross_entropy handles the softmax internally for stability
    loss = mx.mean(mnn.losses.cross_entropy(logits, y))
    return loss

# In MLX, we create a function that computes both loss and gradients
loss_and_grad_fn = mnn.value_and_grad(model, loss_fn)

# 4. The Training Loop (Epochs)
epochs = 400
print("Starting Training...")

for epoch in range(epochs):
    # Compute loss and gradients
    loss, grads = loss_and_grad_fn(model, X_train_mx, y_train_mx)
    
    # Update model parameters using the optimizer
    optimizer.update(model, grads)
    
    # MLX evaluates lazily, so we MUST call mx.eval to execute the update
    mx.eval(model.parameters(), optimizer.state)
    
    # Print progress every 20 epochs
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:3d} | Loss: {loss.item():.4f}")

# 5. Evaluate and Structure Output
print("\nTraining Complete. Evaluating on Test Set...")

# Forward pass on test set (get logits, apply softmax to get probabilities)
test_logits = model(X_test_mx)
test_probs = mnn.softmax(test_logits, axis=1)

probs_np = np.array(test_probs[:5])
true_labels_idx = y_test[:5]

results = []
for i in range(5):
    results.append({
        "True Label": target_names[true_labels_idx[i]].capitalize(),
        "Head 0 (Setosa)": f"{probs_np[i, 0]:.4f}",
        "Head 1 (Versicolor)": f"{probs_np[i, 1]:.4f}",
        "Head 2 (Virginica)": f"{probs_np[i, 2]:.4f}"
    })

print("\nOutput Mapping: Head 0 -> Setosa | Head 1 -> Versicolor | Head 2 -> Virginica\n")
print(pd.DataFrame(results).to_string(index=False))

Starting Training...
Epoch  50 | Loss: 0.0502
Epoch 100 | Loss: 0.0469
Epoch 150 | Loss: 0.0467
Epoch 200 | Loss: 0.0467
Epoch 250 | Loss: 0.0466
Epoch 300 | Loss: 0.0466
Epoch 350 | Loss: 0.0466
Epoch 400 | Loss: 0.0465

Training Complete. Evaluating on Test Set...

Output Mapping: Head 0 -> Setosa | Head 1 -> Versicolor | Head 2 -> Virginica

True Label Head 0 (Setosa) Head 1 (Versicolor) Head 2 (Virginica)
Versicolor          0.0000              0.9999             0.0001
    Setosa          1.0000              0.0000             0.0000
 Virginica          0.0000              0.0000             1.0000
Versicolor          0.0000              0.9981             0.0019
Versicolor          0.0000              0.9984             0.0016


In [9]:


# Prepare all 4 features and 3 species
X_full = df.iloc[:, :4].values
y_full = df['target'].values

# Scale and split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_full, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_test, dtype=torch.float32)

# Define MLP architecture
class IrisMLP(nn.Module):
    def __init__(self):
        super(IrisMLP, self).__init__()
        self.hidden = nn.Linear(4, 8)  # 4 inputs to 8 hidden nodes
        self.output = nn.Linear(8, 3)  # 8 hidden nodes to 3 output heads
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1) # Forces 0 to 1 activation across the 3 heads

    def forward(self, x):
        x = self.relu(self.hidden(x))
        logits = self.output(x)
        probs = self.softmax(logits)
        return probs

model = IrisMLP()

# Perform a forward pass on the test set (untrained model for demonstration)
with torch.no_grad():
    predictions = model(X_tensor)
    
print("Sample Output from the 3 Heads (Values between 0 and 1):")
print(np.round(predictions[:5].numpy(), 4))

Sample Output from the 3 Heads (Values between 0 and 1):
[[0.2056 0.3863 0.4081]
 [0.1757 0.449  0.3753]
 [0.1702 0.3567 0.4731]
 [0.207  0.3972 0.3957]
 [0.1905 0.3637 0.4459]]


In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# 1. Load Data
iris = load_iris(as_frame=True)
df = iris.frame

# Target: Petal Width. Predictors: Sepal Length, Sepal Width, Petal Length
X_reg = df[['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)']].values
y_reg = df['petal width (cm)'].values

# 2. Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# 3. Scale the features (Crucial for MLPs)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reg)
X_test_scaled = scaler.transform(X_test_reg)

# 4. Define and Train the MLP Regressor
# Architecture: 3 inputs -> 16 hidden nodes -> 8 hidden nodes -> 1 linear output
mlp_regressor = MLPRegressor(
    hidden_layer_sizes=(16, 8), 
    activation='relu',          # Non-linear activation for hidden layers
    solver='SGD',              
    max_iter=1000,              # Epochs
    random_state=42
)

mlp_regressor.fit(X_train_scaled, y_train_reg)

# 5. Predict and evaluate
y_pred_reg = mlp_regressor.predict(X_test_scaled)
mse = mean_squared_error(y_test_reg, y_pred_reg)

print(f"Mean Squared Error: {mse:.4f}")

# Compare first 5 predictions against actual values
comparison = pd.DataFrame({'Actual': y_test_reg[:5], 'Predicted': y_pred_reg[:5]})
print("\nFirst 5 Predictions:")
print(comparison.round(3))

Mean Squared Error: 0.0696

First 5 Predictions:
   Actual  Predicted
0     1.2      1.524
1     0.3      0.313
2     2.3      2.669
3     1.5      1.343
4     1.4      1.622
